# Step 2: All-Match Master Join Audit

This notebook audits the Step 2 master join table created by `python main.py step2`.

Expected contract:

- the all-match table includes every tracking frame from every match with both raw files
- tracking-source columns use the `t.` prefix
- event-source columns use the `e.` prefix
- `t.match_id` identifies the source match for every tracking row
- only `PASS`, `BALL TOUCH`, `AERIAL`, `TACKLE`, `BALL RECOVERY`, `FOUL`, and `TAKEON` are attached as events
- each selected event is attached to its nearest tracking frame in the same period
- when multiple selected events choose the same tracking frame, only the closest event is kept
- tracking rows without an attached selected event have all `e.*` fields set to `"no event"`
- raw `ball` is unpacked into `t.ball_x`, `t.ball_y`, and `t.ball_z`
- raw nested `data` player rows are unpacked into `t.player_XX_*` slot columns
- `nearest_timestamp_distance_sec` is the event-to-frame QA column


In [31]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from driblab.etl.master_join import DEFAULT_STEP2_EVENT_TYPES
from driblab.etl.pipeline import load_tracking_file

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_BASE_DIR = PROJECT_ROOT / "data" / "processed" / "model_base"
MASTER_JOIN_PATH = MODEL_BASE_DIR / "master_join_table.parquet"
SUMMARY_PATH = MODEL_BASE_DIR / "master_join_summary.csv"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print(MASTER_JOIN_PATH)


/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/processed/model_base/master_join_table.parquet


## 1. Load the all-match output

This reads the schema, summary file, and a sample from the generated all-match Parquet table.

In [32]:
assert MASTER_JOIN_PATH.exists(), f"Missing {MASTER_JOIN_PATH}. Run: python main.py step2"
assert SUMMARY_PATH.exists(), f"Missing {SUMMARY_PATH}. Run: python main.py step2"

parquet_file = pq.ParquetFile(MASTER_JOIN_PATH)
schema_columns = parquet_file.schema_arrow.names
summary = pd.read_csv(SUMMARY_PATH)
sample = parquet_file.read_row_group(0).to_pandas().head(20)

print(f"Rows in Parquet metadata: {parquet_file.metadata.num_rows:,}")
print(f"Columns: {len(schema_columns):,}")
display(summary.head())
display(sample)


Rows in Parquet metadata: 1,986,630
Columns: 146


,match_id,tracking_rows,raw_events,selected_events,event_type_names,matched_events,master_join_rows,master_join_event_rows,median_abs_nearest_timestamp_distance_sec,p95_abs_nearest_timestamp_distance_sec
0,678949,62016,1461,1320,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1112,62016,1112,0.0240,0.047
1,679026,59502,1424,1295,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1225,59502,1225,0.0250,0.047
2,679053,61798,1526,1440,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1277,61798,1277,0.0250,0.047
3,679072,59968,1504,1377,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1258,59968,1258,0.0255,0.048
4,679075,57962,1501,1409,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1348,57962,1348,0.0250,0.048


,t.match_id,t.match_clock,t.frame,t.Videotimestamp,t.period,t.cam,t.ball_x,t.ball_y,t.ball_z,t.player_count,t.visible_player_count,t.player_01_team_id,t.player_01_id,t.player_01_x,t.player_01_y,t.player_01_visible,t.player_02_team_id,t.player_02_id,t.player_02_x,t.player_02_y,t.player_02_visible,t.player_03_team_id,t.player_03_id,t.player_03_x,t.player_03_y,t.player_03_visible,t.player_04_team_id,t.player_04_id,t.player_04_x,t.player_04_y,t.player_04_visible,t.player_05_team_id,t.player_05_id,t.player_05_x,t.player_05_y,t.player_05_visible,t.player_06_team_id,t.player_06_id,t.player_06_x,t.player_06_y,t.player_06_visible,t.player_07_team_id,t.player_07_id,t.player_07_x,t.player_07_y,t.player_07_visible,t.player_08_team_id,t.player_08_id,t.player_08_x,t.player_08_y,t.player_08_visible,t.player_09_team_id,t.player_09_id,t.player_09_x,t.player_09_y,t.player_09_visible,t.player_10_team_id,t.player_10_id,t.player_10_x,t.player_10_y,...,t.player_16_team_id,t.player_16_id,t.player_16_x,t.player_16_y,t.player_16_visible,t.player_17_team_id,t.player_17_id,t.player_17_x,t.player_17_y,t.player_17_visible,t.player_18_team_id,t.player_18_id,t.player_18_x,t.player_18_y,t.player_18_visible,t.player_19_team_id,t.player_19_id,t.player_19_x,t.player_19_y,t.player_19_visible,t.player_20_team_id,t.player_20_id,t.player_20_x,t.player_20_y,t.player_20_visible,t.player_21_team_id,t.player_21_id,t.player_21_x,t.player_21_y,t.player_21_visible,t.player_22_team_id,t.player_22_id,t.player_22_x,t.player_22_y,t.player_22_visible,e.match_id,e.period_id,e.min,e.sec,e.x,e.y,e.outcome,e.qualifiers,e.possession_id,e.xa,e.xg,e.xt,e.x_start,e.y_start,e.x_end,e.y_end,e.milisec,e.event.id,e.event.event_type_id,e.event.event_type_name,e.team.team_id,e.team.team_name,e.player.player_id,e.player.player_name,nearest_timestamp_distance_sec
0,678949,"[0,1]",0,1.0,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,NaN
1,678949,"[0,1]",1,1.1,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,no event,NaN
2,678949,"[0,1]",2,1.2,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,678949.0,1.0,0.0,1.0,49.0,50.0,True,"[{""qualifier_id"":140,""qualifier_name"":""Pass En...",2.0,0.0,0.0,0.0,49.0,50.0,20.0,62.0,211.0,549113076.0,1.0,PASS,1925.0,Sunderland,69716.0,Granit Xhaka,0.011
3,678949,"[0,1]",3,1.3,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,Na

## 2. Check prefixes and source columns

The final schema should be `t.*`, `e.*`, and `nearest_timestamp_distance_sec`. The source match must be present as `t.match_id`.

In [33]:
t_cols = [col for col in schema_columns if col.startswith("t.")]
e_cols = [col for col in schema_columns if col.startswith("e.")]
other_cols = [col for col in schema_columns if not col.startswith("t.") and not col.startswith("e.")]

prefix_check = pd.DataFrame({
    "check": [
        "t.* columns",
        "e.* columns",
        "other columns",
        "has t.match_id",
        "only allowed other column",
    ],
    "value": [
        len(t_cols),
        len(e_cols),
        other_cols,
        "t.match_id" in schema_columns,
        other_cols == ["nearest_timestamp_distance_sec"],
    ],
})

display(prefix_check)
print("Tracking columns:")
print(t_cols)
print("Event columns:")
print(e_cols)

assert "t.match_id" in schema_columns
assert other_cols == ["nearest_timestamp_distance_sec"]
assert all(col.startswith("t.") for col in t_cols)
assert all(col.startswith("e.") for col in e_cols)


,check,value
0,t.* columns,121
1,e.* columns,24
2,other columns,[nearest_timestamp_distance_sec]
3,has t.match_id,True
4,only allowed other column,True


Tracking columns:
['t.match_id', 't.match_clock', 't.frame', 't.Videotimestamp', 't.period', 't.cam', 't.ball_x', 't.ball_y', 't.ball_z', 't.player_count', 't.visible_player_count', 't.player_01_team_id', 't.player_01_id', 't.player_01_x', 't.player_01_y', 't.player_01_visible', 't.player_02_team_id', 't.player_02_id', 't.player_02_x', 't.player_02_y', 't.player_02_visible', 't.player_03_team_id', 't.player_03_id', 't.player_03_x', 't.player_03_y', 't.player_03_visible', 't.player_04_team_id', 't.player_04_id', 't.player_04_x', 't.player_04_y', 't.player_04_visible', 't.player_05_team_id', 't.player_05_id', 't.player_05_x', 't.player_05_y', 't.player_05_visible', 't.player_06_team_id', 't.player_06_id', 't.player_06_x', 't.player_06_y', 't.player_06_visible', 't.player_07_team_id', 't.player_07_id', 't.player_07_x', 't.player_07_y', 't.player_07_visible', 't.player_08_team_id', 't.player_08_id', 't.player_08_x', 't.player_08_y', 't.player_08_visible', 't.player_09_team_id', 't.player_0

## 3. Check row counts against raw tracking files

The all-match master join must have one row per raw tracking frame.

In [34]:
raw_tracking_rows = 0
raw_tracking_by_match = []
for path in sorted(RAW_DATA_DIR.glob("*_tracking_data.jsonl")):
    match_id = path.name.split("_", 1)[0]
    event_path = RAW_DATA_DIR / f"{match_id}_events.json"
    if not event_path.exists():
        continue
    _, frames = load_tracking_file(path)
    raw_tracking_rows += len(frames)
    raw_tracking_by_match.append({"match_id": int(match_id), "raw_tracking_rows": len(frames)})

raw_tracking_by_match = pd.DataFrame(raw_tracking_by_match)
count_check = summary.merge(raw_tracking_by_match, on="match_id", how="left")
count_check["rows_match_raw_tracking"] = count_check["master_join_rows"].eq(count_check["raw_tracking_rows"])

print(f"Raw tracking rows: {raw_tracking_rows:,}")
print(f"Parquet rows: {parquet_file.metadata.num_rows:,}")
display(count_check.head())

assert parquet_file.metadata.num_rows == raw_tracking_rows
assert count_check["rows_match_raw_tracking"].all()


Raw tracking rows: 1,986,630
Parquet rows: 1,986,630


,match_id,tracking_rows,raw_events,selected_events,event_type_names,matched_events,master_join_rows,master_join_event_rows,median_abs_nearest_timestamp_distance_sec,p95_abs_nearest_timestamp_distance_sec,raw_tracking_rows,rows_match_raw_tracking
0,678949,62016,1461,1320,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1112,62016,1112,0.0240,0.047,62016,True
1,679026,59502,1424,1295,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1225,59502,1225,0.0250,0.047,59502,True
2,679053,61798,1526,1440,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1277,61798,1277,0.0250,0.047,61798,True
3,679072,59968,1504,1377,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1258,59968,1258,0.0255,0.048,59968,True
4,679075,57962,1501,1409,PASS|BALL TOUCH|AERIAL|TACKLE|BALL RECOVERY|FO...,1348,57962,1348,0.0250,0.048,57962,True


## 4. Check event labels and no-event rows

Only the selected Step 2 event types should appear. All other tracking rows should be labelled `no event` in `e.event.event_type_name`.

In [35]:
label_series = pd.read_parquet(MASTER_JOIN_PATH, columns=["e.event.event_type_name"])["e.event.event_type_name"]
observed_events = sorted(set(label_series.dropna()) - {"no event"})
allowed_events = sorted(DEFAULT_STEP2_EVENT_TYPES)

label_check = pd.DataFrame({
    "metric": [
        "allowed selected events",
        "observed attached events",
        "observed matches allowed",
        "rows with selected event",
        "rows with no event",
    ],
    "value": [
        allowed_events,
        observed_events,
        observed_events == allowed_events,
        int(label_series.ne("no event").sum()),
        int(label_series.eq("no event").sum()),
    ],
})

display(label_check)
assert observed_events == allowed_events
assert int(label_series.ne("no event").sum()) == int(summary["master_join_event_rows"].sum())


,metric,value
0,allowed selected events,"[AERIAL, BALL RECOVERY, BALL TOUCH, FOUL, PASS..."
1,observed attached events,"[AERIAL, BALL RECOVERY, BALL TOUCH, FOUL, PASS..."
2,observed matches allowed,True
3,rows with selected event,42437
4,rows with no event,1944193


## 5. Check nearest timestamp distances

Rows with selected events should have a timestamp distance. Rows with no event should have missing distance.

In [36]:
qa = pd.read_parquet(
    MASTER_JOIN_PATH,
    columns=["e.event.event_type_name", "nearest_timestamp_distance_sec"],
)

event_distance_ok = qa.loc[
    qa["e.event.event_type_name"].ne("no event"),
    "nearest_timestamp_distance_sec",
].notna().all()
no_event_distance_ok = qa.loc[
    qa["e.event.event_type_name"].eq("no event"),
    "nearest_timestamp_distance_sec",
].isna().all()

distance_check = pd.DataFrame({
    "check": [
        "event rows have distance",
        "no-event rows have missing distance",
        "median distance seconds",
        "p95 distance seconds",
        "max distance seconds",
    ],
    "value": [
        bool(event_distance_ok),
        bool(no_event_distance_ok),
        qa["nearest_timestamp_distance_sec"].median(),
        qa["nearest_timestamp_distance_sec"].quantile(0.95),
        qa["nearest_timestamp_distance_sec"].max(),
    ],
})

display(distance_check)
assert bool(event_distance_ok)
assert bool(no_event_distance_ok)


,check,value
0,event rows have distance,True
1,no-event rows have missing distance,True
2,median distance seconds,0.025
3,p95 distance seconds,0.048
4,max distance seconds,66.315


## 6. Show event rows from the all-match table

This displays rows where a selected event was attached, with every column from the all-match master join table. It reads row groups until it finds 20 event rows, so it avoids loading the full table into memory.

In [37]:
event_preview_chunks = []
for row_group_index in range(parquet_file.num_row_groups):
    chunk = parquet_file.read_row_group(row_group_index).to_pandas()
    event_chunk = chunk[chunk["e.event.event_type_name"].ne("no event")]
    if not event_chunk.empty:
        event_preview_chunks.append(event_chunk)
    if sum(len(part) for part in event_preview_chunks) >= 20:
        break

preview_events = pd.concat(event_preview_chunks, ignore_index=True).head(20)

print(f"Columns shown: {len(preview_events.columns):,}")
display(preview_events)


Columns shown: 146


,t.match_id,t.match_clock,t.frame,t.Videotimestamp,t.period,t.cam,t.ball_x,t.ball_y,t.ball_z,t.player_count,t.visible_player_count,t.player_01_team_id,t.player_01_id,t.player_01_x,t.player_01_y,t.player_01_visible,t.player_02_team_id,t.player_02_id,t.player_02_x,t.player_02_y,t.player_02_visible,t.player_03_team_id,t.player_03_id,t.player_03_x,t.player_03_y,t.player_03_visible,t.player_04_team_id,t.player_04_id,t.player_04_x,t.player_04_y,t.player_04_visible,t.player_05_team_id,t.player_05_id,t.player_05_x,t.player_05_y,t.player_05_visible,t.player_06_team_id,t.player_06_id,t.player_06_x,t.player_06_y,t.player_06_visible,t.player_07_team_id,t.player_07_id,t.player_07_x,t.player_07_y,t.player_07_visible,t.player_08_team_id,t.player_08_id,t.player_08_x,t.player_08_y,t.player_08_visible,t.player_09_team_id,t.player_09_id,t.player_09_x,t.player_09_y,t.player_09_visible,t.player_10_team_id,t.player_10_id,t.player_10_x,t.player_10_y,...,t.player_16_team_id,t.player_16_id,t.player_16_x,t.player_16_y,t.player_16_visible,t.player_17_team_id,t.player_17_id,t.player_17_x,t.player_17_y,t.player_17_visible,t.player_18_team_id,t.player_18_id,t.player_18_x,t.player_18_y,t.player_18_visible,t.player_19_team_id,t.player_19_id,t.player_19_x,t.player_19_y,t.player_19_visible,t.player_20_team_id,t.player_20_id,t.player_20_x,t.player_20_y,t.player_20_visible,t.player_21_team_id,t.player_21_id,t.player_21_x,t.player_21_y,t.player_21_visible,t.player_22_team_id,t.player_22_id,t.player_22_x,t.player_22_y,t.player_22_visible,e.match_id,e.period_id,e.min,e.sec,e.x,e.y,e.outcome,e.qualifiers,e.possession_id,e.xa,e.xg,e.xt,e.x_start,e.y_start,e.x_end,e.y_end,e.milisec,e.event.id,e.event.event_type_id,e.event.event_type_name,e.team.team_id,e.team.team_name,e.player.player_id,e.player.player_name,nearest_timestamp_distance_sec
0,678949,"[0,1]",2,1.20,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,678949.0,1.0,0.0,1.0,49.0,50.0,True,"[{""qualifier_id"":140,""qualifier_name"":""Pass En...",2.0,0.0,0.0,0.0,49.0,50.0,20.0,62.0,211.0,549113076.0,1.0,PASS,1925.0,Sunderland,69716.0,Granit Xhaka,0.011
1,678949,"[0,5]",48,5.72,1,"[[-29.51,-36.37],[151.2,-93.0],[40.64,71.35],[...",24.02,41.51,0.000,22,10,1925,1221508.0,55.79,20.41,True,1925,1305867.0,15.35,33.73,False,1925,152211.0,73.38,23.46,False,1925,1161497.0,41.46,24.62,True,1925,1747257.0,53.70,42.63,True,1925,1636680.0,57.50,63.38,True,1925,69716.0,52.42,29.53,True,1925,1166824.0,38.47,33.70,True,1925,1373299.0,69.11,12.10,False,1925,1349368.0,50.95,20.95,...,1946,1158319.0,71.26,36.83,False,1946,1170483.0,75.51,24.17,False,1946,1170494.0,50.23,41.64,True,1946,1166550.0,62.98,10.26,False,1946,1237721.0,95.78,33.16,False,1946,80743.0,65.12,30.96,False,1946,1308395.0,69.29,29.67,False,678949.0,1.0,0.0,5.0,19.9,61.5,True,"[{""qualifier_id"":141,""qualifier_name"":""Pass En...",2.0,0.0,0.0,0.013,20.0,62.0,70.0,45.0,671.0,549113077.0,1.0,PASS,1925.0,Sunderland,1305867.0,Robin Roefs,0.029
2,678949,"[0,9]",90,9.92,1,"[[11.0,-33.86],[157.1,-51.26],[43.48,75.42],[7...",NaN,NaN,NaN,22,6,1925,1221508.0,69.03,27.01,False,1925,1305867.0,17.29,33.19,False,1925,152211.0,75.93,21.90,False,1925,1161497.0,49.81,21.94,True,1925,1747257.0,65.63,35.87,False,1925,1636680.0,67.85,47.82,False,1925,69716.0,61.25,24.46,True,1925,1166824.0,43.90,34.60,True,1925,1373299.0,69.97,11.94,False,1925,1349368.0,63.42,17.91,...,1946,1158319.0,77.11,29.93,False,1946,1170483.0,78.28,22.52,False,1946,1170494.0,58.77,40.84,True,1946,1166550.0,66.78,16.12,False,1946,1237721.0,96.66,32.69,False,1946,80743.0,71.91,29.75,False,1946,1308395.0,71.56,29.16,False,678949.0,1.0,0.0,9.0,29.7,54.9,True,"[{""qualifier_id"":233,""qualif